![Cabecera](../../assets/cabecera_sql.png)

# SQL en Python: WHERE

Bueno, pues ya empezamos con las clausulas WHERE aunque por ahora serán relativamente sencillas. Vamos haciendo las cargas de rigor:

In [1]:
import pandas as pd
import sqlite3

# Conectamos con la base de datos chinook.db
connection = sqlite3.connect("data/chinook.db")

# Obtenemos un cursor que utilizaremos para hacer las queries
cursor_bootcamp = connection.cursor()

Pero antes de empezar...

### Una función muy práctica

Si te fijaste en la sesión anterior repetíamos siempre el mismo código y esquema para hacer una query y luego llevarla a un dataframe. Es en esos casos en los que una función se hace necesaria... por ejemplo:

In [2]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    cursor_bootcamp.execute(query) # Recuerda que sólo funcionará si has llamado cursor_bootcamp
                                    # a tu cursor, si no, cambia el nombre en todo el código de la función

    # Almacena los datos de la query 
    ans = cursor_bootcamp.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in cursor_bootcamp.description]

    return pd.DataFrame(ans,columns=names)

Probemos:

In [3]:
query = '''
SELECT composer
FROM tracks
'''
sql_query(query)

,Composer
0,"Angus Young, Malcolm Young, Brian Johnson"
1,None
2,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho..."
3,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D..."
4,Deaffy & R.A. Smith-Diesel
...,...
3498,None
3499,Franz Schubert
3500,Claudio Monteverdi
3501,Wolfgang Amadeus Mozart


### WHERE

Se usa para filtrar filas como ya sabes, veamos algunos ejemplos de uso:


#### Filtros numéricos

* **Un valor numérico**
    * UnitPrice = 0.99
    * UnitPrice >= 0.99
    * UnitPrice < 0.99


In [4]:
# Escogiendo la tabla tracks canciones que duren más de 6 minutos (360000 milisegundos)
minutos = 6
query = f'''
SELECT name "Nombre canción", Milliseconds "Duración"
FROM tracks
WHERE Milliseconds > {minutos * 60 * 1000}
'''

sql_query(query)

,Nombre canción,Duración
0,Princess of the Dawn,375418
1,Let There Be Rock,366654
2,Overdose,369319
3,Livin' On The Edge,381231
4,You Oughta Know (Alternate),491885
...,...,...
618,Symphony No. 3 Op. 36 for Orchestra and Sopran...,567494
619,"Act IV, Symphony",364296
620,"3 Gymnopédies: No.1 - Lent Et Grave, No.3 - Le...",385506
621,Symphony No. 2: III. Allegro vivace,376510


Podríamos dejarlo además en segundos operando directamente la columna en la primera parte del SELECT:


In [5]:
minutos = 6
query = f'''
SELECT name "Nombre canción", Milliseconds/1000 "Duración"
FROM tracks
WHERE Milliseconds > {minutos * 60 * 1000}
'''

sql_query(query)

,Nombre canción,Duración
0,Princess of the Dawn,375
1,Let There Be Rock,366
2,Overdose,369
3,Livin' On The Edge,381
4,You Oughta Know (Alternate),491
...,...,...
618,Symphony No. 3 Op. 36 for Orchestra and Sopran...,567
619,"Act IV, Symphony",364
620,"3 Gymnopédies: No.1 - Lent Et Grave, No.3 - Le...",385
621,Symphony No. 2: III. Allegro vivace,376


#### Filtros sobre campos de texto

* **Un valor string**: Name = 'Restless and Wild'
* **string contenido**:
    * strings que empiecen por 'A': Name like 'A%'
    * strings que acaben en 'A': Name like '%A'
    * strings que lleven 'A' en algun punto: Name like '%A%'


  

In [6]:
# Compositores que se llaman Brian
query = '''
SELECT *
FROM tracks
WHERE composer LIKE "%Brian%"
'''
sql_query(query)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,6,Put The Finger On You,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",205662,6713451,0.99
2,7,Let's Get It Up,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",233926,7636561,0.99
3,8,Inject The Venom,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",210834,6852860,0.99
4,9,Snowballed,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",203102,6599424,0.99
5,10,Evil Walks,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",263497,8611245,0.99
6,11,C.O.D.,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",199836,6566314,0.99
7,12,Breaking The Rules,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",263288,8596840,0.99
8,13,Night Of The Long Knives,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",205688,6706347,0.99
9,14,Spellbound,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",270863,8817038,0.99


#### Filtros varios

  
* **Varios valores**: GenreId in (1, 5, 12)  
* **Distinto de**: UnitPrice <> 0.99


In [9]:
# Clientes que vivan en Berlin, Londres o París
query = '''
SELECT FirstName "Nombre", LastName "Apellido", City "Ciudad"
FROM Customers
WHERE City in("Berlin", "London", "Paris")
'''
sql_query(query)

,Nombre,Apellido,Ciudad
0,Hannah,Schneider,Berlin
1,Niklas,Schröder,Berlin
2,Camille,Bernard,Paris
3,Dominique,Lefebvre,Paris
4,Emma,Jones,London
5,Phil,Hughes,London


#### Combinaciones booleanas de condiciones

Por supuesto podemos combinar los filtros ya que no dejan de ser condiciones booleanas al igual que hacemos con los `DataFrame` de pandas.  Complicando un poco más el `WHERE`

In [12]:
# Busquemos las canciones que cuesten más de 0.99 o duren menos de 3 minutos y además de eso en su nombre aparezca la palabra Fire
query = '''
SELECT *
FROM tracks
WHERE (unitprice > 0.99 or Milliseconds < 3*60*1000 and Name LIKE "%fire%")
'''
sql_query(query)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1486,Fire,120,1,1,Jimi Hendrix,164989,5383075,0.99
1,1948,Fire Fire,160,1,3,Clarke/Kilmister/Taylor,164675,5416114,0.99
2,2666,Play With Fire,216,1,1,Nanker Phelge,132022,4265297,0.99
3,2819,Battlestar Galactica: The Story So Far,226,3,18,None,2622250,490750393,1.99
4,2820,Occupation / Precipice,227,3,19,None,5286953,1054423946,1.99
...,...,...,...,...,...,...,...,...,...
212,3363,"There's No Place Like Home, Pt. 2",261,3,21,None,2497956,523748920,1.99
213,3364,"There's No Place Like Home, Pt. 3",261,3,21,None,2582957,486161766,1.99
214,3428,Branch Closing,251,3,22,None,1814855,360331351,1.99
215,3429,The Return,251,3,22,None,1705080,343877320,1.99
